In [26]:
# !pip install

In [27]:
!pip install xgboost

In [38]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split

import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV

import joblib

In [29]:
# Cargar datasets limpios desde la carpeta clean
df_train = pd.read_csv("Data/clean/train_clean.csv")
df_test = pd.read_csv("Data/clean/test_clean.csv")

# Verificación rápida
print(df_train.shape)
print(df_test.shape)

df_train.head()
df_test.head()

C:\Users\Laura\AppData\Local\Temp\ipykernel_10216\565899242.py:2: DtypeWarning: Columns (0: transferred) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("Data/clean/train_clean.csv")


(2642706, 18)
(411642, 18)


C:\Users\Laura\AppData\Local\Temp\ipykernel_10216\565899242.py:3: DtypeWarning: Columns (0: transferred) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv("Data/clean/test_clean.csv")


,family,sales,city,state,type_x,cluster,transactions,dcoilwtico,transferred,sales_log,transactions_log,has_promotion,day_of_week,day,month,year,is_earthquake,is_holiday
0,POULTRY,0.0,Machala,El Oro,D,4,NaN,52.36,True,0.0,NaN,0,Sunday,1,1,2017,0,1
1,BEVERAGES,0.0,Cuenca,Azuay,D,2,NaN,52.36,True,0.0,NaN,0,Sunday,1,1,2017,0,1
2,BEAUTY,0.0,Cuenca,Azuay,D,2,NaN,52.36,True,0.0,NaN,0,Sunday,1,1,2017,0,1
3,BABY CARE,0.0,Cuenca,Azuay,D,2,NaN,52.36,True,0.0,NaN,0,Sunday,1,1,2017,0,1
4,AUTOMOTIVE,0.0,Cuenca,Azuay,D,2,NaN,52.36,True,0.0,NaN,0,Sunday,1,1,2017,0,1


In [30]:
# Aplicar One-Hot Encoding a las variables categóricas
df_train_encoded = pd.get_dummies(df_train, drop_first=True)
df_test_encoded = pd.get_dummies(df_test, drop_first=True)

# Alinear test con train para que tengan las mismas columnas
df_test_encoded = df_test_encoded.reindex(columns=df_train_encoded.columns, fill_value=0)

In [31]:
# Definir variable objetivo con logaritmo y quitar el petróleo (dcoilwtico)
y = np.log1p(df_train_encoded['sales'])
X = df_train_encoded.drop(columns=['sales', 'dcoilwtico'], errors='ignore')

# División Holdout 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [32]:
# Rellena todos los nulos con 0
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("Nulos restantes:", X_train.isnull().sum().sum())

Nulos restantes: 0


In [33]:
# Configuración y entrenamiento
modelo_ridge = Ridge(alpha=1.0)
modelo_ridge.fit(X_train, y_train)

# Predicción y métricas
pred_ridge = modelo_ridge.predict(X_test)
rmse_ridge = np.sqrt(mean_squared_error(y_test, pred_ridge))
mae = mean_absolute_error(y_test, pred_ridge)
r2_ridge = r2_score(y_test, pred_ridge)

print(f"Ridge Regression:")
print(f"RMSE: {rmse_ridge:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R2 Score: {r2_ridge:.4f}")

Ridge Regression:
RMSE: 0.0000
MAE (Mean Absolute Error): 0.0000
R2 Score: 1.0000


In [34]:
# Configuración del modelo
modelo_rf = RandomForestRegressor(
    n_estimators=100, 
    max_depth=10, 
    random_state=42, 
    n_jobs=-1
)

# Entrenamiento
modelo_rf.fit(X_train, y_train)

# Predicción y métricas
y_pred_rf = modelo_rf.predict(X_test)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest finalizado:")
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R2 Score: {r2_rf:.4f}")

Random Forest finalizado:
RMSE: 0.0010
MAE (Mean Absolute Error): 0.0004
R2 Score: 1.0000


In [35]:
# Configuración del modelo XGBoost
modelo_xgb = xgb.XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    random_state=42
)

# Entrenamiento
modelo_xgb.fit(X_train, y_train)

# Predicción
y_pred_xgb = modelo_xgb.predict(X_test)

# Cálculo de toda la información de rendimiento (Métricas)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
mae = mean_absolute_error(y_test, y_pred_xgb)
r2 = r2_score(y_test, y_pred_xgb)

# 5. Visualización de resultados para tu PPT
print("RESULTADOS DETALLADOS - XGBOOST")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R2 Score (Varianza explicada): {r2:.4f}")

RESULTADOS DETALLADOS - XGBOOST
RMSE (Root Mean Squared Error): 0.0124
MAE (Mean Absolute Error): 0.0044
R2 Score (Varianza explicada): 1.0000


In [40]:
# Definir el modelo
xgb_model = xgb.XGBRegressor(random_state=42)

# Definir el espacio de búsqueda (usamos rangos más amplios)
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

# Configurar RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=10, 
    cv=3, 
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

# Entrenar
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies

In [41]:
# Guardar Ridge
joblib.dump(modelo_ridge, "modelos/modelo_ridge.pkl")
print("Ridge guardado correctamente en modelos/")

# Guardar Random Forest
joblib.dump(modelo_rf, "modelos/modelo_random_forest.pkl")
print("Random Forest guardado correctamente en modelos/")

# Guardar el mejor XGBoost del Randomized Search
joblib.dump(random_search.best_estimator_, "modelos/modelo_final_xgboost.pkl")
print("XGBoost optimizado guardado correctamente en modelos/modelo_final_xgboost.pkl")

# Guardar el mejor Random Forest del Randomized Search
joblib.dump(random_search.best_estimator_, "modelos/modelo_rf_optimized.pkl")
print("Random Forest optimizado guardado correctamente en modelos/modelo_rf_optimized.pkl")

print("Modelo guardado correctamente en modelos")

Ridge guardado correctamente en modelos/
Random Forest guardado correctamente en modelos/
XGBoost optimizado guardado correctamente en modelos/modelo_final_xgboost.pkl
Random Forest optimizado guardado correctamente en modelos/modelo_rf_optimized.pkl
Modelo guardado correctamente en modelos
